In [ ]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import gc
import zipfile

from tqdm import tqdm
import shutil

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Loading package

In [ ]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[2]
sys.path.append(str(repo_path))

In [ ]:
from py.utils import verifyDir,verifyFile

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

DATA_PATH = os.getenv('DATA_PATH')
MODEL_PATH = os.getenv('MODEL_PATH')
DATA_PATH, MODEL_PATH

In [ ]:
MODEL_NAME="OneFormer_Swin_Large"
SEG_DATASET="ade20k" # cityscapes

In [ ]:
ADE20K_DIR = f"{DATA_PATH}{SEG_DATASET}/"
UPD4K_DIR = f"{DATA_PATH}/upd4k/"

QSCORE_PATH=f"{DATA_PATH}pp2/Qscores/"
IMAGES_PATH = f"{DATA_PATH}pp2/images/"
SEGMENT_DIR = f"{DATA_PATH}pp2/segmentations/{SEG_DATASET}/{MODEL_NAME}/"
UPD_SEGMENT_DIR = f"{DATA_PATH}UrbanPhysicalDisorder/upd4k/"

In [ ]:
verifyDir(UPD_SEGMENT_DIR)

### Loading data

In [ ]:
from py.datasets import UrbanPhysicalDisorder

uss = UrbanPhysicalDisorder(data_path=DATA_PATH)
uss.generate_dataset(dataset="ade20k_upd4k")

objects_df = uss.get_urban_street_categories()
objects_df

### Merging masks

In [ ]:
import fnmatch

In [ ]:
columns_to_keep = ["image_id", "seg_image_path", "seg_overlay_image_path", "mask_path", "ratio_path"]

In [ ]:
%%time
for current_city in ["Rio De Janeiro"]:
    OUT_DIR = f"{UPD_SEGMENT_DIR}{current_city}/"
    
    verifyDir(OUT_DIR)
    verifyDir(f"{OUT_DIR}/masks/")
    verifyDir(f"{OUT_DIR}/segmented_images/")
    verifyDir(f"{OUT_DIR}/segmented_images_overlay/")
    verifyDir(f"{OUT_DIR}/ratios/")
    
    merge_segment_df = pd.DataFrame()
    manual_segment_df = pd.DataFrame()
    # model_segment_df = pd.DataFrame()
    
    usd_c_dir = f'{UPD4K_DIR}/{current_city}/'
    
    #manual_img_seg_list = np.sort([f for f in glob.glob(f'{usd_c_dir}/SegmentationClass/*.png')])
    with zipfile.ZipFile(f'{UPD4K_DIR}/{current_city}.zip', 'r') as zip_ref:
        # Get all PNG files in SegmentationClass directory
        manual_img_seg_list = np.sort([f'{UPD4K_DIR}{f}' for f in zip_ref.namelist() 
                     if fnmatch.fnmatch(f, f'{current_city}/SegmentationClass/*.png')])
    
    manual_img_seg_ids = [ img.split("/")[-1].replace(".png", "") for img in manual_img_seg_list]
    
    img_seg_list = np.sort([f for f in glob.glob(f'{SEGMENT_DIR}/{current_city}/segmented_images/*.png') ])
    img_seg_ids = [ img.split("/")[-1].replace(".png", "") for img in img_seg_list]
    
    for img_path in tqdm(img_seg_list):
        
        current_id = img_path.split("/")[-1].replace(".JPEG", "").replace(".png", "")
    
        current_image = Image.open(glob.glob(f'{IMAGES_PATH}/{current_city}/{current_id}.JPG')[0]).convert("RGB")
        
        if current_id in manual_img_seg_ids:
            manual_img_seg = Image.open(
                                zipfile.ZipFile(f'{UPD4K_DIR}/{current_city}.zip')
                                .open(f'{current_city}/SegmentationClass/{current_id}.png')
                            ).convert("RGB")
            
            colors_tuple = uss.calculate_unique_colors(manual_img_seg, to_tuple=True)
    
            # Calculate manual masks ratios
            manual_masks = uss.convert_mask_to_matrix(manual_img_seg, colors_tuple, objects_df)
            manual_df = uss.calculate_pixel_ratios(manual_masks, objects_df)
            manual_pivot_df = uss.parse_ratios(current_id, manual_df)
            manual_pivot_df["seg_image_path"] = f"{current_city}/segmented_images/{current_id}.png"
            manual_pivot_df["seg_overlay_image_path"] = f"{current_city}/segmented_images_overlay/{current_id}.png"
            manual_pivot_df["mask_path"] = f"{current_city}/masks/{current_id}.pkl"
            manual_pivot_df["ratio_path"] = f"{current_city}/ratios/{current_id}.csv"
            
            manual_segment_df = pd.concat([manual_segment_df, manual_pivot_df], ignore_index=True)
            manual_segment_df = manual_segment_df[columns_to_keep + [col for col in manual_segment_df.columns if col not in columns_to_keep]].copy()
            manual_segment_df.fillna(0, inplace=True)
    
            # Calculate real masks ratios
            model_masks = joblib.load(glob.glob(f'{SEGMENT_DIR}/{current_city}/masks/{current_id}.pkl')[0])
            
            # merging masks
            merged_masks = np.where(manual_masks != 0, manual_masks, model_masks)
            joblib.dump(merged_masks, f"{UPD_SEGMENT_DIR}{current_city}/masks/{current_id}.pkl")
            ratio_df = uss.calculate_pixel_ratios(merged_masks, objects_df)
            ratio_df.to_csv(f"{UPD_SEGMENT_DIR}{current_city}/ratios/{current_id}.csv", sep=";", index=False)
    
            # merging both images
            ade20k_img_seg = Image.open(glob.glob(f'{SEGMENT_DIR}/{current_city}/segmented_images/{current_id}.png')[0]).convert("RGB")
            merged_img_seg = uss.merge_segmentations(ade20k_img_seg, manual_img_seg)
            merged_image = Image.fromarray(merged_img_seg)
            merged_image.save(f"{UPD_SEGMENT_DIR}{current_city}/segmented_images/{current_id}.png")
    
            # Overlay image
            image_overlay = Image.blend(current_image, merged_image, alpha=0.6)
            image_overlay.save(f"{UPD_SEGMENT_DIR}{current_city}/segmented_images_overlay/{current_id}.png")
    
        else:
            # copying masks
            model_masks = joblib.load(glob.glob(f'{SEGMENT_DIR}/{current_city}/masks/{current_id}.pkl')[0])
            source = glob.glob(f'{SEGMENT_DIR}/{current_city}/masks/{current_id}.pkl')[0]
            destination = f"{UPD_SEGMENT_DIR}{current_city}/masks/{current_id}.pkl"
            shutil.copy(source, destination)
    
            # copying ratios
            source = glob.glob(f'{SEGMENT_DIR}/{current_city}/ratios/{current_id}.csv')[0]
            destination = f"{UPD_SEGMENT_DIR}{current_city}/ratios/{current_id}.csv"
            shutil.copy(source, destination)
    
            # copying image segmentation
            source = glob.glob(f'{SEGMENT_DIR}/{current_city}/segmented_images/{current_id}.png')[0]
            destination = f"{UPD_SEGMENT_DIR}{current_city}/segmented_images/{current_id}.png"
            shutil.copy(source, destination)
    
            # copying image overlay
            source = glob.glob(f'{SEGMENT_DIR}/{current_city}/segmented_images_overlay/{current_id}.png')[0]
            destination = f"{UPD_SEGMENT_DIR}{current_city}/segmented_images_overlay/{current_id}.png"
            shutil.copy(source, destination)
    
            # read ratios to append
            ratio_df = pd.read_csv(f'{SEGMENT_DIR}/{current_city}/ratios/{current_id}.csv', sep=";", low_memory=False)
    
        # model segmentations
        # model_df = uss.calculate_pixel_ratios(model_masks, objects_df)
        # model_pivot_df = uss.parse_ratios(current_id, model_df)
        # model_segment_df = pd.concat([model_segment_df, model_pivot_df], ignore_index=True)
        # model_segment_df = model_segment_df[columns_to_keep + [col for col in model_segment_df.columns if col not in columns_to_keep]].copy()
        # model_segment_df.fillna(0, inplace=True)
    
        # merge segmentations
        merge_pivot_df = uss.parse_ratios(current_id, ratio_df)
        merge_pivot_df["seg_image_path"] = f"{current_city}/segmented_images/{current_id}.png"
        merge_pivot_df["seg_overlay_image_path"] = f"{current_city}/segmented_images_overlay/{current_id}.png"
        merge_pivot_df["mask_path"] = f"{current_city}/masks/{current_id}.pkl"
        merge_pivot_df["ratio_path"] = f"{current_city}/ratios/{current_id}.csv"
        
        merge_segment_df = pd.concat([merge_segment_df, merge_pivot_df], ignore_index=True)
        merge_segment_df = merge_segment_df[columns_to_keep + [col for col in merge_segment_df.columns if col not in columns_to_keep]].copy()
        merge_segment_df.fillna(0, inplace=True)
    
    manual_segment_df.to_csv(f"{UPD_SEGMENT_DIR}{current_city}/upd4k_segmentations.csv", sep=";", index=False)
    merge_segment_df.to_csv(f"{UPD_SEGMENT_DIR}{current_city}/segmentations.csv", sep=";", index=False)

### Summary

In [ ]:
manual_segment_df = pd.read_csv(f"{UPD_SEGMENT_DIR}{current_city}/upd4k_segmentations.csv", sep=";", low_memory=False)
merge_segment_df = pd.read_csv(f"{UPD_SEGMENT_DIR}{current_city}/segmentations.csv", sep=";", low_memory=False)

In [ ]:
manual_segment_df.shape, merge_segment_df.shape#, model_segment_df.shape

In [ ]:
set(merge_segment_df.columns) - set(manual_segment_df.columns)#, set(merge_segment_df.columns) - set(model_segment_df.columns)

In [ ]:
metric="safety"

In [ ]:
%%time
data_df = pd.read_csv(f"{QSCORE_PATH}scores.csv", sep=";", low_memory=False)
manual_segmentation_df = pd.merge(data_df, manual_segment_df, on="image_id", how="inner")
manual_segmentation_df.sort_values(by=metric, inplace=True, ascending=False)
manual_segmentation_df

### Object Presence

#### All samples

In [ ]:
manual_segmentation_df.info()

In [ ]:
feature_presence = uss.features_presence(manual_segmentation_df.iloc[:, 17:])
feature_presence[0] = feature_presence[0]*100
uss.print_object_present(feature_presence, fig_size=(16,20))

#### Positive samples

In [ ]:
feature_presence = uss.features_presence(manual_segmentation_df[manual_segmentation_df[metric]>=6].iloc[:, 17:])
uss.print_object_present(feature_presence, fig_size=(16,20))

#### Negative samples

In [ ]:
feature_presence = uss.features_presence(manual_segmentation_df[manual_segmentation_df[metric]<4].iloc[:, 17:])
uss.print_object_present(feature_presence, fig_size=(16,20))

### Boxplots

#### All samples

In [ ]:
uss.print_object_boxplot(manual_segmentation_df.iloc[:, 17:]*100, fig_size=(25, 20))

#### Positive samples

In [ ]:
uss.print_object_boxplot(manual_segmentation_df[manual_segmentation_df[metric]>=6].iloc[:, 17:]*100, fig_size=(25, 20))

#### Negative samples

In [ ]:
uss.print_object_boxplot(manual_segmentation_df[manual_segmentation_df[metric]<4].iloc[:, 17:]*100, fig_size=(25, 20))